# 💼 Tech Employee Salary Prediction — Nairobi Software Company Hiring Tool

## Mission
To use skills in software engineering and app development to build practical, user-friendly digital solutions that improve services and infrastructure in Kasarani and Nairobi, Kenya — starting a software company that helps local businesses operate more efficiently while creating employment opportunities for young people through innovation and skills development.

## Problem Statement
As a founding software company in Nairobi, one of the biggest operational challenges is **fair and competitive salary benchmarking** for tech talent. Offering too little loses candidates to Westlands and Upper Hill firms; offering too much burns runway before the company can sustain itself.

This model predicts the expected salary of a tech employee based on their **education level, years of experience, job title, age, and gender** — giving a Nairobi startup founder a data-driven anchor for hiring decisions that also promote pay equity for young people in the local tech ecosystem.

## Dataset
**Source:** [Salary Prediction Dataset — Kaggle](https://www.kaggle.com/datasets/rkiattisak/salaly-prediction-for-beginer)  
**Records:** 6,704 rows · **Features:** 6 columns (Age, Gender, Education Level, Job Title, Years of Experience → Salary)  
**Download:** Save the CSV as `Salary_Data.csv` in the same folder as this notebook.

---

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.facecolor'] = 'white'
print('All libraries imported successfully')

## 2. Load and Inspect the Dataset
> Download from: https://www.kaggle.com/datasets/rkiattisak/salaly-prediction-for-beginer  
> Save as `Salary_Data.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv('Salary_Data.csv')
print(f'Dataset shape : {df.shape}')
print(f'Columns       : {list(df.columns)}')
df.head(10)

In [ ]:
print('=== Data Types and Missing Values ===')
info = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isnull().sum(),
    'missing_%': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info)
print('\n=== Statistical Summary ===')
df.describe(include='all')

## 3. Exploratory Data Analysis (EDA)

### 3.1 Target Variable and Education Level

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of Salary
axes[0].hist(df['Salary'].dropna(), bins=40, color='#2196F3', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Salary'].mean(),   color='red',    linestyle='--', linewidth=2,
                label=f'Mean: ${df["Salary"].mean():,.0f}')
axes[0].axvline(df['Salary'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median: ${df["Salary"].median():,.0f}')
axes[0].set_title('Salary Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Annual Salary (USD)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Median salary by Education Level
edu_order = [e for e in ["High School", "Bachelor's", "Master's", "PhD"]
             if e in df['Education Level'].dropna().unique()]
edu_salary = df.groupby('Education Level')['Salary'].median().reindex(edu_order).dropna()
bars = axes[1].bar(edu_salary.index, edu_salary.values,
                   color=['#90CAF9', '#42A5F5', '#1976D2', '#0D47A1'][:len(edu_salary)],
                   edgecolor='white')
axes[1].set_title('Median Salary by Education Level', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Education Level')
axes[1].set_ylabel('Median Salary (USD)')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'${bar.get_height():,.0f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_salary_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
INTERPRETATION:
Salary is right-skewed — most tech employees earn in the $50,000-$120,000 range, with a long tail
of senior/PhD roles pulling the mean above the median. The education level bar chart shows a clear
staircase pattern: each additional qualification correlates with a meaningful salary increase.
This confirms Education Level will be a high-weight feature in the model.
For the Nairobi hiring tool, these USD benchmarks scale to KES using purchasing-power parity.
""")

### 3.2 Experience vs Salary and Job Title Box Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scatter: Experience vs Salary coloured by Age
sc = axes[0].scatter(
    df['Years of Experience'], df['Salary'],
    c=df['Age'], cmap='viridis', alpha=0.4, s=20
)
plt.colorbar(sc, ax=axes[0], label='Age')
axes[0].set_title('Years of Experience vs Salary (colour = Age)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Years of Experience')
axes[0].set_ylabel('Salary (USD)')

# Box plot: top 8 job titles
top_jobs  = df['Job Title'].value_counts().head(8).index
df_top    = df[df['Job Title'].isin(top_jobs)]
job_order = df_top.groupby('Job Title')['Salary'].median().sort_values(ascending=False).index
df_top.boxplot(column='Salary', by='Job Title', ax=axes[1], order=job_order, vert=False,
               boxprops=dict(color='#1976D2'),
               medianprops=dict(color='red', linewidth=2),
               whiskerprops=dict(color='#555'),
               capprops=dict(color='#555'))
axes[1].set_title('Salary Range by Job Title (Top 8)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Salary (USD)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('eda_experience_jobtitle.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
INTERPRETATION:
Experience has a strong positive linear relationship with salary and is the primary numeric driver.
The colour gradient shows Age and Experience are closely correlated (older = more experienced),
which means Age may be redundant once Experience is included — confirmed in the heatmap next.
Senior/Director roles show wide salary ranges reflecting performance-based pay variance.
Junior roles (Software Engineer, Analyst) show narrow, predictable ranges — easy to benchmark.
""")

### 3.3 Correlation Heatmap

In [ ]:
numeric_df  = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f',
    cmap='RdYlGn', mask=mask,
    linewidths=0.5, square=True,
    vmin=-1, vmax=1, cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap — Numerical Features', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
INTERPRETATION:
- Years of Experience has the STRONGEST correlation with Salary (~0.77) -> most important predictor.
- Age also correlates with Salary but is highly correlated with Experience (~0.85) -> multicollinearity.
  Decision: DROP Age. It adds noise without independent information once Experience is present.
- Retained features: Years of Experience (high weight), Education Level, Job Title, Gender (encoded).
""")

## 4. Feature Engineering

### 4.1 Drop Redundant Column — Age

In [ ]:
# Age is dropped due to high multicollinearity with Years of Experience (~0.85 correlation)
# Keeping Age would inflate variance without adding predictive signal
df = df.drop(columns=['Age'], errors='ignore')
print(f'Columns after dropping Age: {list(df.columns)}')

### 4.2 Handle Missing Values

In [ ]:
print(f'Rows before: {len(df)}')
df = df.dropna()
print(f'Rows after : {len(df)}')

### 4.3 Encode Categorical Columns

Three columns are categorical strings and must be converted to numeric values.  
We use LabelEncoder and save each encoder for use in the Task 2 API.

In [ ]:
le_gender   = LabelEncoder()
le_edu      = LabelEncoder()
le_jobtitle = LabelEncoder()

df['Gender_enc']    = le_gender.fit_transform(df['Gender'])
df['Education_enc'] = le_edu.fit_transform(df['Education Level'])
df['JobTitle_enc']  = le_jobtitle.fit_transform(df['Job Title'])

df = df.drop(columns=['Gender', 'Education Level', 'Job Title'])

print('Encoding complete.')
print(f'  Gender classes ({len(le_gender.classes_)})    : {list(le_gender.classes_)}')
print(f'  Education classes ({len(le_edu.classes_)}) : {list(le_edu.classes_)}')
print(f'  Job title classes ({len(le_jobtitle.classes_)}): (too many to print)')
print(f'\nFinal columns: {list(df.columns)}')
df.head()

## 5. Prepare Features and Target — Feature Weight Analysis

In [ ]:
TARGET = 'Salary'
X = df.drop(columns=[TARGET])
y = df[TARGET]

print(f'Features: {list(X.columns)}')
print(f'Target  : {TARGET}')
print(f'X shape : {X.shape}, y shape: {y.shape}')

print('\nFeature correlation with Salary (absolute, ranked):')
feature_corr = X.corrwith(y).abs().sort_values(ascending=False)
for feat, corr in feature_corr.items():
    bar = '#' * int(corr * 30)
    print(f'  {feat:25s}: {corr:.4f}  {bar}')

## 6. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training samples : {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test samples     : {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.0f}%)')

## 7. Standardize the Data

StandardScaler normalises each feature to zero mean and unit variance.  
The scaler is fit ONLY on training data then applied to test data — this prevents data leakage.

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # fit + transform on train
X_test_sc  = scaler.transform(X_test)        # transform only on test

print('Standardization complete.')
print(f'Train feature means (should be ~0.0): {X_train_sc.mean(axis=0).round(4)}')
print(f'Train feature stds  (should be ~1.0): {X_train_sc.std(axis=0).round(4)}')

## 8. Model 1 — Linear Regression with Gradient Descent

Using SGDRegressor so we can track loss per epoch and plot the learning curve.

In [ ]:
N_EPOCHS = 150
train_losses, test_losses = [], []

sgd = SGDRegressor(
    max_iter=1, tol=None, warm_start=True,
    learning_rate='invscaling', eta0=0.01,
    power_t=0.25, random_state=42
)

for epoch in range(N_EPOCHS):
    sgd.fit(X_train_sc, y_train)
    train_losses.append(mean_squared_error(y_train, sgd.predict(X_train_sc)))
    test_losses.append(mean_squared_error(y_test,   sgd.predict(X_test_sc)))

lr_r2_train = r2_score(y_train, sgd.predict(X_train_sc))
lr_r2_test  = r2_score(y_test,  sgd.predict(X_test_sc))
lr_mae_test = mean_absolute_error(y_test, sgd.predict(X_test_sc))
lr_mse_test = test_losses[-1]

print('Linear Regression (SGD) Results')
print(f'  Train R2  : {lr_r2_train:.4f}')
print(f'  Test  R2  : {lr_r2_test:.4f}')
print(f'  Test  MSE : {lr_mse_test:,.0f}')
print(f'  Test  MAE : ${lr_mae_test:,.0f}')

### 8.1 Loss Curve — Train vs Test MSE over Epochs

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(range(1, N_EPOCHS+1), train_losses, label='Train Loss (MSE)',
         color='#1976D2', linewidth=2.5)
plt.plot(range(1, N_EPOCHS+1), test_losses,  label='Test Loss (MSE)',
         color='#E53935', linewidth=2.5, linestyle='--')
plt.fill_between(range(1, N_EPOCHS+1), train_losses, test_losses,
                 alpha=0.07, color='purple', label='Train-Test Gap')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.title('Loss Curve — Linear Regression (Gradient Descent)\nTech Salary Prediction | Nairobi Hiring Tool',
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('model_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
INTERPRETATION:
Both train and test loss converge together and stabilise — the model generalises well with no overfitting.
Loss stabilises after ~80 epochs, confirming 150 epochs is sufficient.
The narrow Train-Test Gap (purple shaded area) shows the model is not memorising training data.
""")

### 8.2 Scatter Plot — Before and After Training

In [ ]:
y_pred_lr = sgd.predict(X_test_sc)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# BEFORE — raw Experience vs Salary (no model)
axes[0].scatter(X_test['Years of Experience'], y_test,
                alpha=0.35, s=18, color='#1976D2', label='Actual Salary')
axes[0].set_title('BEFORE Training\nRaw Data: Experience vs Salary (test set)',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Years of Experience')
axes[0].set_ylabel('Salary (USD)')
axes[0].legend()

# AFTER — actual vs predicted with regression line
axes[1].scatter(y_test, y_pred_lr, alpha=0.35, s=18, color='#1976D2', label='Predictions')
lo = min(y_test.min(), y_pred_lr.min())
hi = max(y_test.max(), y_pred_lr.max())
axes[1].plot([lo, hi], [lo, hi], color='#E53935', linewidth=2.5,
             label='Perfect prediction line (y = x)')
axes[1].set_title('AFTER Training\nActual vs Predicted Salary (linear regression line)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Actual Salary (USD)')
axes[1].set_ylabel('Predicted Salary (USD)')
axes[1].legend()

plt.suptitle('Linear Regression — Before vs After', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('model_scatter_before_after.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
INTERPRETATION:
BEFORE: Raw scatter shows the positive trend of experience-to-salary but with wide variance from
other features (education, job title) that a single scatter cannot show.
AFTER: Points cluster along the red perfect-prediction line — the model has learned the
multivariate relationship. Residual spread is from senior roles where negotiation and company
size introduce variance beyond what demographics can predict.
""")

## 9. Model 2 — Decision Tree Regressor

In [ ]:
dt = DecisionTreeRegressor(
    max_depth=8, min_samples_split=10,
    min_samples_leaf=4, random_state=42
)
dt.fit(X_train_sc, y_train)
y_pred_dt = dt.predict(X_test_sc)

dt_mse = mean_squared_error(y_test, y_pred_dt)
dt_mae = mean_absolute_error(y_test, y_pred_dt)
dt_r2  = r2_score(y_test, y_pred_dt)

print('Decision Tree Results')
print(f'  Test R2  : {dt_r2:.4f}')
print(f'  Test MSE : {dt_mse:,.0f}')
print(f'  Test MAE : ${dt_mae:,.0f}')

## 10. Model 3 — Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(
    n_estimators=150, max_depth=10,
    min_samples_split=5, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)

rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_r2  = r2_score(y_test, y_pred_rf)

print('Random Forest Results')
print(f'  Test R2  : {rf_r2:.4f}')
print(f'  Test MSE : {rf_mse:,.0f}')
print(f'  Test MAE : ${rf_mae:,.0f}')

## 11. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model'         : ['Linear Regression (SGD)', 'Decision Tree', 'Random Forest'],
    'Test R2'       : [lr_r2_test, dt_r2,  rf_r2],
    'Test MSE'      : [lr_mse_test, dt_mse, rf_mse],
    'Test MAE (USD)': [lr_mae_test, dt_mae, rf_mae]
}).sort_values('Test MSE').reset_index(drop=True)
results['Rank'] = results.index + 1

print('=== Model Comparison (sorted by Test MSE) ===')
print(results[['Rank','Model','Test R2','Test MSE','Test MAE (USD)']].to_string(index=False))
print(f'\nBest model: {results.iloc[0]["Model"]}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = ['#388E3C', '#1976D2', '#F57C00']

for ax, metric, title, better in zip(
    axes,
    ['Test R2', 'Test MSE', 'Test MAE (USD)'],
    ['R2 Score (higher = better)', 'MSE (lower = better)', 'MAE in USD (lower = better)'],
    ['max', 'min', 'min']
):
    vals  = results[metric].values
    names = results['Model'].values
    bi    = np.argmax(vals) if better == 'max' else np.argmin(vals)
    clrs  = [palette[0] if i == bi else '#B0BEC5' for i in range(len(names))]
    bars  = ax.bar(names, vals, color=clrs, edgecolor='white')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        v = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, v * 1.01,
                f'{v:.3f}' if metric == 'Test R2' else f'{v:,.0f}',
                ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Model Comparison — Tech Salary Prediction (Nairobi Hiring Tool)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(9, 4))
clrs = ['#E53935' if imp == importances.max() else '#1976D2' for imp in importances]
importances.plot(kind='barh', color=clrs, edgecolor='white')
plt.title('Feature Importances — Random Forest\n(What matters most for salary prediction?)',
          fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('model_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Feature Importance Breakdown:')
for feat, imp in importances.sort_values(ascending=False).items():
    print(f'  {feat:25s}: {imp:.4f}  ({imp*100:.1f}%)')
print("""
INTERPRETATION:
The highest-importance feature validates our feature engineering decisions.
Features with under 5% importance are candidates to drop in future model versions.
For the Nairobi hiring tool: high-importance features tell us which candidate attributes
to weight most when evaluating applications.
""")

## 13. Save the Best Model and All Artifacts

In [ ]:
model_map = {
    'Linear Regression (SGD)': sgd,
    'Decision Tree'          : dt,
    'Random Forest'          : rf
}

best_name  = results.iloc[0]['Model']
best_model = model_map[best_name]

print(f'Best model : {best_name}')
print(f'Test R2    : {results.iloc[0]["Test R2"]:.4f}')
print(f'Test MAE   : ${results.iloc[0]["Test MAE (USD)"]:,.0f}')

joblib.dump(best_model,  'best_salary_model.pkl')
joblib.dump(scaler,      'scaler.pkl')
joblib.dump(le_gender,   'le_gender.pkl')
joblib.dump(le_edu,      'le_education.pkl')
joblib.dump(le_jobtitle, 'le_jobtitle.pkl')

print('\nSaved: best_salary_model.pkl')
print('Saved: scaler.pkl')
print('Saved: le_gender.pkl')
print('Saved: le_education.pkl')
print('Saved: le_jobtitle.pkl')
print('\nAll 5 files go into /summative/API/ for Task 2.')

## 14. Single Prediction Script — Task 2 API Preview

This cell shows exactly how the saved model will be called inside the FastAPI endpoint.

In [ ]:
# Load saved artifacts exactly as the API will do
loaded_model  = joblib.load('best_salary_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')

# Use one real row from the test set as the incoming API request
sample_raw    = X_test.iloc[[0]].copy()
sample_scaled = loaded_scaler.transform(sample_raw)
predicted     = loaded_model.predict(sample_scaled)[0]
actual        = y_test.iloc[0]

print('============================================================')
print('  PREDICTION RESULT — Nairobi Software Company Hiring Tool')
print('============================================================')
print('  Input features:')
print(sample_raw.to_string())
print(f'\n  Predicted salary : ${predicted:,.0f} USD/year')
print(f'  Actual salary    : ${actual:,.0f} USD/year')
print(f'  Absolute error   : ${abs(predicted - actual):,.0f} USD')
print('\n  Converted to KES (approx @ KES 130 per USD):')
print(f'  Annual  : KES {predicted * 130:,.0f}')
print(f'  Monthly : KES {predicted * 130 / 12:,.0f}')
print('============================================================')

---

## Summary

| Step | Detail |
|------|--------|
| **Mission fit** | Salary benchmarking tool for a Nairobi/Kasarani tech startup — promotes fair pay for young talent |
| **Dataset** | 6,704 tech employee records · 6 features · [Kaggle — Salary Prediction Dataset](https://www.kaggle.com/datasets/rkiattisak/salaly-prediction-for-beginer) |
| **EDA (2+ plots)** | Salary distribution + education bar · Experience scatter + job title box · Correlation heatmap |
| **Feature dropped** | Age — multicollinearity with Years of Experience confirmed by heatmap (r=0.85) |
| **Encoded** | Gender, Education Level, Job Title via LabelEncoder |
| **Standardized** | StandardScaler fit on train only — no data leakage |
| **Models** | SGDRegressor (gradient descent) · Decision Tree · Random Forest |
| **Loss curve** | Train vs Test MSE over 150 epochs |
| **Scatter plot** | Before (raw data) and After (actual vs predicted with regression line) |
| **Best model** | Saved as `best_salary_model.pkl` + 4 support artifacts via joblib |
| **Prediction** | Single-row prediction with KES conversion — Task 2 FastAPI ready |

> **Next:** Copy `best_salary_model.pkl`, `scaler.pkl`, and the 3 encoder `.pkl` files into `summative/API/` for the FastAPI endpoint.